In [ ]:
#0. 작업 준비
import numpy as np
import torch
import matplotlib.pyplot as plt
from dateutil.tz import EPOCH

from torch.utils import data
from torchvision import datasets,transforms, utils
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

USE_MPS = torch.backends.mps.is_available()
DEVICE=torch.device('mps' if USE_MPS else 'cpu')

# DCGAN 모델 생성

In [ ]:
BATCH_SIZE = 128
EPOCHS = 30

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,),(0.5,)) # [-1, 1]
])
tr_ds = datasets.MNIST(root='./data', train=True, transform=transform, download=False)
tr_ds_loader = data.DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

In [ ]:
Z_DIM = 3
IMG_C = 1
IMG_SIZE = 28

In [ ]:
D = nn.Sequential(
    nn.Conv2d(IMG_C, 64, 4, 2, 1, bias=False), #[64, 14, 14]
    nn.LeakyReLU(0.2, inplace=True),
    nn.Conv2d(64, 128, 4, 2, 1, bias=False), #[128, 7, 7]
    nn.BatchNorm2d(128),
    nn.LeakyReLU(0.2, inplace=True),
    nn.Flatten(),
    nn.Linear(128 * 7 * 7, 1),
    nn.Sigmoid()
)

In [ ]:
G = nn.Sequential(
    nn.Linear(Z_DIM, 128*7*7),
    nn.BatchNorm1d(128*7*7),
    nn.ReLU(),
    nn.Unflatten(1, (128, 7, 7)), #[128, 7, 7]
    nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False), #[64, 14, 14]
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.ConvTranspose2d(64, IMG_C, 4, 2, 1, bias=False), #[1, 28, 28]
    nn.Tanh()
)

In [ ]:
G = G.to(DEVICE)
D = D.to(DEVICE)

In [ ]:
criterion = nn.BCELoss()
d_opt = optim.Adam(D.parameters(), lr=0.0002)
g_opt = optim.Adam(G.parameters(), lr=0.0002)

In [ ]:
z_noise = torch.randn(BATCH_SIZE, Z_DIM, device=DEVICE)
for epoch in range(EPOCHS):
    for i, (x, _) in enumerate(tr_ds_loader):
        data = x.to(DEVICE)
        # data.size[0] 배치 사이즈
        r_label = torch.ones(BATCH_SIZE, 1, device=DEVICE)
        f_label = torch.zeros(BATCH_SIZE, 1, device=DEVICE)

        z = torch.randn(BATCH_SIZE, Z_DIM, device=DEVICE)
        f_img = G(z)

        real_out = D(data)
        d_loss_real = criterion(real_out, r_label)
        real_sc = real_out.detach()

        fake_out = D(f_img.detach())
        d_loss_fake = criterion(fake_out, f_label)
        fake_sc = fake_out.detach()

        d_loss = d_loss_real + d_loss_fake

        g_opt.zero_grad()
        d_opt.zero_grad()
        d_loss.backward()
        d_opt.step()

        fake_out_ck_g = D(f_img)
        g_loss = criterion(fake_out_ck_g, r_label)

        g_opt.zero_grad()
        d_opt.zero_grad()
        g_loss.backward()
        g_opt.step()
    print(f'{epoch+1} 회, d_loss: {d_loss.item()}, g_loss: {g_loss.item()} D(G(z)): {fake_sc.mean().item()} D(x): {real_sc.mean().item()}')

    with torch.no_grad():
        G.eval()
        v_img = G(z_noise) #(64, 1, 28, 28)
        v_img = (v_img+1)/2

        f, a = plt.subplots(8, 8, figsize=(8, 8))
        for i, ax in enumerate(a.flat):
            img = v_img[i].cpu().permute(1, 2, 0).numpy()
            ax.imshow(img.squeeze(), cmap='gray')
            ax.axis('off')
        plt.tight_layout()
        plt.show()
        G.train()

# 패션 데이터를 이용하여 DCGAN을 구현하시오

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,),(0.5,)) # [-1, 1]
])
tr_ds = datasets.FashionMNIST(root='./data', train=True, transform=transform, download=False)
tr_ds_loader = data.DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

In [ ]:
Z_DIM = 3
IMG_C = 1
IMG_SIZE = 28

In [ ]:
D = nn.Sequential(
    nn.Conv2d(IMG_C, 64, 4, 2, 1, bias=False),  #[64, 14, 14]
    nn.LeakyReLU(0.2, inplace=True),
    nn.Conv2d(64, 128, 4, 2, 1, bias=False),  #[128, 7, 7]
    nn.BatchNorm2d(128),
    nn.LeakyReLU(0.2, inplace=True),
    nn.Flatten(),
    nn.Linear(128 * 7 * 7, 1),
    nn.Sigmoid()
)
G = nn.Sequential(
    nn.Linear(Z_DIM, 128 * 7 * 7),
    nn.BatchNorm1d(128 * 7 * 7),
    nn.ReLU(),
    nn.Unflatten(1, (128, 7, 7)),  #[128, 7, 7]
    nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),  #[64, 14, 14]
    nn.BatchNorm2d(64),
    nn.ReLU(),
    nn.ConvTranspose2d(64, IMG_C, 4, 2, 1, bias=False),  #[1, 28, 28]
    nn.Tanh()
)

In [ ]:
G = G.to(DEVICE)
D = D.to(DEVICE)
criterion = nn.BCELoss()
d_opt = optim.Adam(D.parameters(), lr=0.0002)
g_opt = optim.Adam(G.parameters(), lr=0.0002)

In [ ]:
z_noise = torch.randn(BATCH_SIZE, Z_DIM, device=DEVICE)
for epoch in range(EPOCHS):
    for i, (x, _) in enumerate(tr_ds_loader):
        data = x.to(DEVICE)
        # data.size[0] 배치 사이즈
        r_label = torch.ones(BATCH_SIZE, 1, device=DEVICE)
        f_label = torch.zeros(BATCH_SIZE, 1, device=DEVICE)

        z = torch.randn(BATCH_SIZE, Z_DIM, device=DEVICE)
        f_img = G(z)

        real_out = D(data)
        d_loss_real = criterion(real_out, r_label)
        real_sc = real_out.detach()

        fake_out = D(f_img.detach())
        d_loss_fake = criterion(fake_out, f_label)
        fake_sc = fake_out.detach()

        d_loss = d_loss_real + d_loss_fake

        g_opt.zero_grad()
        d_opt.zero_grad()
        d_loss.backward()
        d_opt.step()

        fake_out_ck_g = D(f_img)
        g_loss = criterion(fake_out_ck_g, r_label)

        g_opt.zero_grad()
        d_opt.zero_grad()
        g_loss.backward()
        g_opt.step()
    print(
        f'{epoch + 1} 회, d_loss: {d_loss.item()}, g_loss: {g_loss.item()} D(G(z)): {fake_sc.mean().item()} D(x): {real_sc.mean().item()}')

    with torch.no_grad():
        G.eval()
        v_img = G(z_noise)  #(64, 1, 28, 28)
        v_img = (v_img + 1) / 2

        f, a = plt.subplots(8, 8, figsize=(8, 8))
        for i, ax in enumerate(a.flat):
            img = v_img[i].cpu().permute(1, 2, 0).numpy()
            ax.imshow(img.squeeze(), cmap='gray')
            ax.axis('off')
        plt.tight_layout()
        plt.show()
        G.train()